# Alzheimer’s Disease Severity Prediction Using Machine Learning
## CA3 Part B — End-to-End Predictive Analytics (10 Marks)

**Framework:** Knowledge Discovery in Databases (KDD)  
**Dataset:** OASIS cross-sectional clinical/MRI-derived measurements  
**Target:** CDR-based severity group

> **Important:** This notebook supports an educational classification study. It is not a medical diagnostic tool. Run every cell from top to bottom after placing the downloaded dataset in the same folder as this notebook. Results and metrics will be produced from the actual file when you run it; do not report results before running it.


## 1. Problem Identification

The objective is to investigate whether demographic, cognitive, and brain-volume measurements can classify the CDR-based cognitive impairment severity recorded in the OASIS dataset.

This is a **cross-sectional classification task**, not a prediction of future disease progression. The target is derived from the dataset’s Clinical Dementia Rating (CDR):
- `0` → Normal (CDR 0)
- `0.5` → Very mild impairment (CDR 0.5)
- `1` or greater → Mild/moderate dementia group (CDR ≥ 1)

These groups are project-defined and simplify the original CDR scale. Interpret predictions cautiously.


## 2. Data Selection and Dataset Source

Download `oasis_cross-sectional.csv` from the Kaggle dataset below and put it beside this notebook.

- Dataset page: https://www.kaggle.com/datasets/jboysen/mri-and-alzheimers
- Original study resource: https://sites.wustl.edu/oasisbrains/

The commonly shared cross-sectional table contains fields such as Age, Educ, SES, MMSE, CDR, eTIV, nWBV, and ASF. Exact columns can vary by file version, so the loading code checks column names before modelling.


In [ ]:
# 3. Import Libraries
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, ConfusionMatrixDisplay, roc_auc_score
)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42
pd.set_option("display.max_columns", 100)


In [ ]:
# 4. Load Dataset
# Put oasis_cross-sectional.csv in the same folder as this notebook.
# Excel is also supported if your downloaded OASIS file is an .xlsx file.
candidates = [
    Path("oasis_cross-sectional.csv"),
    Path("oasis_cross-sectional.xlsx"),
    Path("oasis_cross-sectional-5708aa0a98d82080.xlsx"),
    Path("oasis_cross-sectional-5708aa0a98d82080.xls"),
]

data_path = next((p for p in candidates if p.exists()), None)
if data_path is None:
    raise FileNotFoundError(
        "Dataset not found. Download oasis_cross-sectional.csv from "
        "https://www.kaggle.com/datasets/jboysen/mri-and-alzheimers "
        "and place it in the same folder as this notebook. "
        "Alternatively, update the candidates list with your exact filename."
    )

if data_path.suffix.lower() == ".csv":
    df = pd.read_csv(data_path)
else:
    df = pd.read_excel(data_path)

# Standardize whitespace in headers; preserve original spelling/case otherwise.
df.columns = [str(c).strip() for c in df.columns]
print("Loaded:", data_path)
print("Shape:", df.shape)
display(df.head())


In [ ]:
# 5. Data Understanding
print("Rows, columns:", df.shape)
print("\\nColumn names:")
print(df.columns.tolist())
print("\\nData types and non-null counts:")
display(df.info())
print("\\nMissing values:")
display(df.isna().sum().sort_values(ascending=False).to_frame("missing_count"))
print("\\nDuplicate rows:", df.duplicated().sum())
display(df.describe(include="all").T)


## 6. Data Preparation and KDD Preprocessing

The notebook standardizes common OASIS header variants. It removes duplicate rows and records with no usable CDR target. Missing predictor values are imputed inside each modelling pipeline, so the imputer is fitted only on the training data.

**Leakage prevention:** CDR itself, subject identifiers, MRI identifiers, and other direct identifiers are not used as input predictors. Because this is a small cross-sectional dataset, results may vary with the train/test split. A single test split is used for final comparison and should be interpreted cautiously.


In [ ]:
# Normalize common header variants to canonical names
def canonicalize(name):
    key = str(name).strip().lower().replace("_", " ").replace(".", " ")
    key = " ".join(key.split())
    aliases = {
        "m/f": "gender", "m f": "gender", "gender": "gender", "sex": "gender",
        "educ": "education", "education": "education",
        "ses": "ses", "mmse": "mmse", "cdr": "cdr",
        "etiv": "etiv", "nw bv": "nwbv", "nwbv": "nwbv",
        "asf": "asf", "age": "age",
        "id": "id", "subject id": "subject_id", "subject.id": "subject_id",
        "hand": "hand", "delay": "delay", "mr delay": "mr_delay",
        "group": "group"
    }
    return aliases.get(key, key.replace(" ", "_"))

df = df.copy()
df.columns = [canonicalize(c) for c in df.columns]
print("Canonical columns:", df.columns.tolist())

if "cdr" not in df.columns:
    raise ValueError(
        "Could not find a CDR target column. Available columns are: "
        + ", ".join(df.columns)
        + ". Select an OASIS table containing CDR, or revise the target definition."
    )

df = df.drop_duplicates().copy()
df["cdr"] = pd.to_numeric(df["cdr"], errors="coerce")
df = df.dropna(subset=["cdr"]).copy()

# CDR severity group: 0 / 0.5 / >=1
def cdr_group(value):
    if np.isclose(value, 0.0):
        return "Normal (CDR 0)"
    elif np.isclose(value, 0.5):
        return "Very mild (CDR 0.5)"
    elif value >= 1.0:
        return "Mild/moderate (CDR >= 1)"
    return np.nan

df["severity"] = df["cdr"].apply(cdr_group)
df = df.dropna(subset=["severity"]).copy()

print("Cleaned shape:", df.shape)
print("Severity class counts:")
display(df["severity"].value_counts().to_frame("records"))


In [ ]:
# 7. Exploratory Data Analysis (EDA)
ax = df["severity"].value_counts().reindex(
    ["Normal (CDR 0)", "Very mild (CDR 0.5)", "Mild/moderate (CDR >= 1)"]
).plot(kind="bar", rot=15)
ax.set_title("Distribution of CDR-Based Severity Groups")
ax.set_xlabel("Severity group")
ax.set_ylabel("Number of records")
plt.tight_layout()
plt.show()

numeric_candidates = [c for c in ["age", "education", "ses", "mmse", "etiv", "nwbv", "asf"] if c in df.columns]
if numeric_candidates:
    display(df.groupby("severity")[numeric_candidates].median(numeric_only=True).T)
    df[numeric_candidates].hist(figsize=(12, 8), bins=15)
    plt.suptitle("Distributions of Numeric Features")
    plt.tight_layout()
    plt.show()

if len(numeric_candidates) >= 2:
    corr = df[numeric_candidates].corr(numeric_only=True)
    plt.figure(figsize=(8, 6))
    plt.imshow(corr, aspect="auto")
    plt.xticks(range(len(corr.columns)), corr.columns, rotation=45, ha="right")
    plt.yticks(range(len(corr.index)), corr.index)
    plt.colorbar(label="Correlation")
    plt.title("Numeric Feature Correlation Matrix")
    plt.tight_layout()
    plt.show()


## 8. Feature Selection and Train/Test Split

Candidate predictors are selected only from columns available in the dataset. `CDR` is excluded because it defines the target. Identifiers and non-informative fields such as handedness are excluded. The model uses the available demographic, cognitive, and MRI-derived measures.

**Limitation:** MMSE is another cognitive assessment and may be strongly associated with CDR. The resulting model estimates agreement with the recorded CDR grouping; it does not establish causation or clinical validity.


In [ ]:
# Candidate features that may be present in this OASIS version
numeric_preferred = ["age", "education", "ses", "mmse", "etiv", "nwbv", "asf", "mr_delay", "delay"]
categorical_preferred = ["gender"]

numeric_features = [c for c in numeric_preferred if c in df.columns]
categorical_features = [c for c in categorical_preferred if c in df.columns]
feature_columns = numeric_features + categorical_features

if not feature_columns:
    raise ValueError("No supported predictor columns found. Review the dataset's column names.")

X = df[feature_columns].copy()
y = df["severity"].copy()

# Drop columns that are entirely missing
X = X.dropna(axis=1, how="all")
numeric_features = [c for c in numeric_features if c in X.columns]
categorical_features = [c for c in categorical_features if c in X.columns]
feature_columns = list(X.columns)

print("Features used:", feature_columns)
print("Target classes:", sorted(y.unique()))
print("Class counts:")
display(y.value_counts())

# Stratification requires enough records in each class
class_counts = y.value_counts()
if class_counts.min() < 2:
    raise ValueError("At least two records per severity class are needed for a stratified split.")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)
print("Train size:", len(X_train), "| Test size:", len(X_test))


In [ ]:
# 9. Preprocessing Pipelines
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

transformers = []
if numeric_features:
    transformers.append(("numeric", numeric_transformer, numeric_features))
if categorical_features:
    transformers.append(("categorical", categorical_transformer, categorical_features))

preprocessor = ColumnTransformer(transformers=transformers, remainder="drop")


## 10. Predictive Modelling

Five models are compared: Logistic Regression, Decision Tree, Random Forest, K-Nearest Neighbours (KNN), and Support Vector Machine (SVM). Preprocessing is included in each pipeline to prevent training/test leakage.


In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=3000, class_weight="balanced", random_state=RANDOM_STATE),
    "Decision Tree": DecisionTreeClassifier(class_weight="balanced", random_state=RANDOM_STATE, min_samples_leaf=2),
    "Random Forest": RandomForestClassifier(
        n_estimators=300, class_weight="balanced", random_state=RANDOM_STATE, min_samples_leaf=2
    ),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "SVM": SVC(class_weight="balanced", probability=True, random_state=RANDOM_STATE)
}

pipelines = {
    name: Pipeline(steps=[("preprocess", preprocessor), ("model", model)])
    for name, model in models.items()
}

results = []
fitted_models = {}
for name, pipe in pipelines.items():
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    fitted_models[name] = pipe
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision (macro)": precision_score(y_test, pred, average="macro", zero_division=0),
        "Recall (macro)": recall_score(y_test, pred, average="macro", zero_division=0),
        "F1-score (macro)": f1_score(y_test, pred, average="macro", zero_division=0)
    })

results_df = pd.DataFrame(results).sort_values("F1-score (macro)", ascending=False).reset_index(drop=True)
display(results_df.style.format({c: "{:.3f}" for c in results_df.columns if c != "Model"}))


In [ ]:
# 11. Model Comparison
ax = results_df.set_index("Model")[["Accuracy", "Precision (macro)", "Recall (macro)", "F1-score (macro)"]].plot(
    kind="bar", figsize=(12, 6)
)
ax.set_title("Model Performance Comparison")
ax.set_ylabel("Score")
ax.set_ylim(0, 1)
ax.tick_params(axis="x", rotation=25)
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

best_model_name = results_df.iloc[0]["Model"]
best_model = fitted_models[best_model_name]
best_pred = best_model.predict(X_test)
print("Best model by macro F1-score:", best_model_name)
print("\\nClassification report:")
print(classification_report(y_test, best_pred, zero_division=0))


In [ ]:
# 12. Confusion Matrix
ConfusionMatrixDisplay.from_predictions(
    y_test, best_pred, xticks_rotation=25, cmap="Blues", colorbar=False
)
plt.title(f"Confusion Matrix — {best_model_name}")
plt.tight_layout()
plt.show()


In [ ]:
# 13. ROC-AUC (when supported and meaningful)
# Multiclass one-vs-rest ROC-AUC is reported only if the test split includes all classes
# and the model exposes class probabilities.
try:
    class_order = list(best_model.named_steps["model"].classes_)
    if hasattr(best_model, "predict_proba") and len(np.unique(y_test)) == len(class_order):
        probabilities = best_model.predict_proba(X_test)
        auc = roc_auc_score(y_test, probabilities, multi_class="ovr", average="macro", labels=class_order)
        print(f"Macro one-vs-rest ROC-AUC: {auc:.3f}")
    else:
        print("ROC-AUC not reported: probability scores or all test classes are unavailable.")
except Exception as exc:
    print("ROC-AUC could not be computed for this split:", exc)


## 14. Feature Importance and Interpretation

Permutation importance measures how much the selected model's F1-score changes when one input feature is shuffled. It is model-agnostic and operates on the original input columns. With a small dataset, treat these rankings as exploratory rather than definitive.


In [ ]:
try:
    perm = permutation_importance(
        best_model, X_test, y_test, n_repeats=20, random_state=RANDOM_STATE,
        scoring="f1_macro"
    )
    importance_df = pd.DataFrame({
        "Feature": X_test.columns,
        "Mean importance": perm.importances_mean,
        "Std": perm.importances_std
    }).sort_values("Mean importance", ascending=False)
    display(importance_df)
    ax = importance_df.sort_values("Mean importance").plot(
        x="Feature", y="Mean importance", kind="barh", legend=False, figsize=(8, 5)
    )
    ax.set_title("Permutation Feature Importance")
    ax.set_xlabel("Decrease in macro F1 when shuffled")
    plt.tight_layout()
    plt.show()
except Exception as exc:
    print("Permutation importance could not be calculated:", exc)


## 15. Results and Discussion

After running the notebook, use the generated comparison table, classification report, and confusion matrix to discuss:
1. Which model achieved the highest macro F1-score.
2. Whether performance is balanced across the three severity groups.
3. Which groups are most often confused with one another.
4. Which features appear useful according to permutation importance.
5. The effects of missing data, limited sample size, and class imbalance.

Do not claim that a model is clinically reliable based on this exercise. The dataset is relatively small, the labels are simplified, and validation on independent external data would be needed before any real-world use.


## 16. Conclusion

This notebook implements an end-to-end KDD workflow: problem definition, data selection, understanding, cleaning, exploratory analysis, feature preparation, model development, evaluation, and interpretation. Complete this section after running the notebook and replace general statements with your actual model metrics and observations.

## References
1. OASIS Brains, Washington University in St. Louis: https://sites.wustl.edu/oasisbrains/
2. Boysen, J. “MRI and Alzheimers” dataset, Kaggle: https://www.kaggle.com/datasets/jboysen/mri-and-alzheimers
3. Morris, J. C. (1993). The Clinical Dementia Rating (CDR): Current version and scoring rules. *Neurology*, 43(11), 2412–2414.
